# Kernel Optimization - make the same math run faster on the GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanhnd61-vr/modelopt-101/blob/main/examples/05_kernel_optimization/kernel_optimization.ipynb)

**Goal.** Take one operation - a **matrix multiply**, the workhorse of every neural net -
and write our own GPU kernel for it in [Triton](https://triton-lang.org/) that runs as fast
as PyTorch's built-in `torch.matmul`. The math does not change at all; only *how* the
hardware executes it does.

This is a **different axis** from the other four exercises. Quantization, KD, pruning and
NAS change *what* the model computes. Kernel optimization changes *how fast* a fixed
computation runs - same inputs, same outputs, fewer microseconds.

The parts to focus on are:

1. **Tiling / blocking** - split the output into blocks and reuse data in fast on-chip memory.
2. **Program / grid layout** - how Triton program ids map to output tiles.
3. **Memory access** - coalesced loads and reuse, why tiling cuts slow global-memory traffic.

At the end we **record correctness, latency and throughput (TFLOP/s)** of our kernel vs.
PyTorch eager across a few matrix sizes:

| matrix size | torch.matmul | our Triton kernel |
|-------------|--------------|-------------------|
| 512-4096    | reference    | same result, comparable speed |

**Runtime:** set `Runtime -> Change runtime type -> GPU`, then `Runtime -> Run all`.
A full run is about a minute on a Colab GPU.

## 1. Setup

Triton ships **inside** PyTorch, so on Colab there is nothing to `pip install`. We just
need a **GPU runtime** (`Runtime -> Change runtime type -> GPU`).

In [ ]:
import time
import numpy as np
import torch
import triton
import triton.language as tl

torch.manual_seed(0)
np.random.seed(0)

assert torch.cuda.is_available(), "Set Runtime -> Change runtime type -> GPU"
device = torch.device("cuda")
print("device:", torch.cuda.get_device_name(0))
print("triton:", triton.__version__)

# float16 inputs: matmul is what GPU tensor cores are built for, and fp16 is the
# realistic precision used for inference on the edge.
DTYPE = torch.float16

## 2. The operation: a matrix multiply

We compute $C = A \times B$ with $A$ of shape $(M, K)$ and $B$ of shape $(K, N)$. Each
output element is a dot product over the shared dimension $K$:

$$ C_{m,n} = \sum_{k=0}^{K-1} A_{m,k}\, B_{k,n} $$

That is $2 M N K$ floating-point operations (one multiply + one add per term). The numbers
below are what every kernel must produce; the only question is how fast.

In [ ]:
M, N, K = 1024, 1024, 1024
a = torch.randn(M, K, device=device, dtype=DTYPE)
b = torch.randn(K, N, device=device, dtype=DTYPE)

ref = torch.matmul(a, b)
print("A:", tuple(a.shape), " B:", tuple(b.shape), " C:", tuple(ref.shape))
print("FLOPs per matmul:", 2 * M * N * K)

## 3. Why tiling - the memory wall

A GPU can do arithmetic far faster than it can read from its main (global) memory. A naive
matmul - one thread walking a full row of $A$ and a full column of $B$ per output element -
re-reads the same rows and columns from slow global memory over and over. It becomes
**memory-bound**: the math units sit idle waiting for data.

**Tiling** fixes this. Split the output $C$ into small `BLOCK_M x BLOCK_N` tiles. To compute
one tile, stream over $K$ in chunks of `BLOCK_K`: load a `BLOCK_M x BLOCK_K` piece of $A$ and
a `BLOCK_K x BLOCK_N` piece of $B$ into fast on-chip memory **once**, multiply them, and
accumulate. Every value loaded gets reused `BLOCK_N` (resp. `BLOCK_M`) times before it is
thrown away, so global-memory traffic drops by roughly the block size.

```
        N                         one output tile needs a strip of A
   +---------+                    and a strip of B, streamed K-chunk
   |  . T .  |  <- BLOCK_M x       at a time into on-chip memory:
 M |  . i .  |     BLOCK_N tile
   |  . l .  |        C[tile] += A[strip] @ B[strip]
   +---------+                 ^ accumulate over K in BLOCK_K steps
```

## 4. The tiled matmul kernel 🔑

This is the heart of the exercise. One Triton **program** (the unit that runs on the GPU)
computes **one `BLOCK_M x BLOCK_N` output tile**. Read it in three passes, matching the three
ideas above.

**(a) Program / grid layout.** We launch a 1-D grid of `cdiv(M,BLOCK_M) * cdiv(N,BLOCK_N)`
programs - one per output tile. The `pid -> (pid_m, pid_n)` mapping is *grouped* instead of
plain row-major: programs that run together cover a small `GROUP_M x` square of tiles rather
than a long row. Neighboring tiles then share the same strips of $A$/$B$, so those strips stay
hot in the GPU's L2 cache -> **better memory access**.

**(b) Tiling / blocking.** The `for k` loop walks the shared dimension in `BLOCK_K` steps. Each
step loads one tile of $A$ and one of $B$ and does `tl.dot` (a tile-level matrix multiply on the
tensor cores), accumulating into `acc`. `acc` lives in registers/SRAM - fast on-chip memory -
for the whole loop.

**(c) Memory access.** `tl.arange` + strides build *contiguous* address ranges, so the loads
are **coalesced** (adjacent threads read adjacent addresses - the pattern GPU memory wants).
The `mask=` arguments let the block sizes not divide the matrix evenly without reading
out of bounds.

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,          # pointers to A, B, C in GPU memory
    M, N, K,                       # matrix dimensions
    stride_am, stride_ak,          # how many elements to step for +1 row / +1 col of A
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    # ---- (a) program/grid layout: which output tile is THIS program? ----
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    # TODO: map pid -> (pid_m, pid_n) with GROUPED ordering for L2 reuse (placeholder = plain row-major):
    #       num_pid_in_group = GROUP_M * num_pid_n
    #       group_id = pid // num_pid_in_group; first_pid_m = group_id * GROUP_M
    #       group_size_m = min(num_pid_m - first_pid_m, GROUP_M)
    #       pid_m = first_pid_m + (pid % group_size_m)
    #       pid_n = (pid % num_pid_in_group) // group_size_m
    pid_m = pid // num_pid_n
    pid_n = pid % num_pid_n

    # ---- starting row/col offsets of this tile, and the K offsets ----
    offs_m = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offs_n = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    # ---- (b) tiling: stream over K, accumulate in fast on-chip memory ----
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_K, other=0.0)
        # TODO: accumulate the tile product on the tensor cores -> acc += tl.dot(a, b)
        acc += 0.0
        a_ptrs += BLOCK_K * stride_ak                # slide the tiles along K
        b_ptrs += BLOCK_K * stride_bk

    # ---- (c) write the tile back, masking the ragged edges ----
    offs_cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    # TODO: mask the ragged edges -> c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    c_mask = (offs_cm[:, None] < M + BLOCK_M) & (offs_cn[None, :] < N + BLOCK_N)
    tl.store(c_ptrs, acc.to(c_ptr.dtype.element_ty), mask=c_mask)

## 5. A thin Python wrapper

The wrapper allocates the output, computes the grid size, and launches the kernel. These
**block sizes are the knobs** that decide how much data each program keeps on-chip - try
changing them in the last section.

In [ ]:
def triton_matmul(a, b, BLOCK_M=64, BLOCK_N=64, BLOCK_K=32, GROUP_M=8):
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "inner dimensions must match"
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # 1-D grid: one program per BLOCK_M x BLOCK_N output tile
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),)
    matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K, GROUP_M=GROUP_M,
    )
    return c

## 6. Correctness first

A fast kernel that computes the wrong thing is worthless. Before timing anything, check our
output against `torch.matmul`. We allow a small tolerance because fp16 arithmetic and a
different summation order give tiny rounding differences - the values, not the bits, must match.

In [ ]:
c_triton = triton_matmul(a, b)
c_torch  = torch.matmul(a, b)

max_err = (c_triton - c_torch).abs().max().item()
ok = torch.allclose(c_triton, c_torch, atol=1e-1, rtol=1e-1)
print(f"max abs difference: {max_err:.4f}")
print("match torch.matmul:", ok)
if not ok:
    print("not correct yet - complete the TODOs in the kernel above, then re-run")

## 7. Benchmark - latency & throughput vs. PyTorch eager

Now the payoff. For each square size we time both kernels and report:

- **latency** - milliseconds per matmul (lower is better);
- **throughput** - $\text{TFLOP/s} = \dfrac{2 M N K}{\text{time}}$, how close we get to the
  hardware's peak (higher is better);
- **speedup** - our throughput relative to `torch.matmul`.

`triton.testing.do_bench` handles GPU warmup and synchronization (a kernel launch is async,
so naive `time.time()` would measure nothing). PyTorch's matmul calls cuBLAS, years of
hand-tuning - landing in the same ballpark from a screenful of Python is the whole point.

In [ ]:
def benchmark(size):
    a = torch.randn(size, size, device=device, dtype=DTYPE)
    b = torch.randn(size, size, device=device, dtype=DTYPE)
    flops = 2 * size ** 3

    ms_triton = triton.testing.do_bench(lambda: triton_matmul(a, b))
    ms_torch  = triton.testing.do_bench(lambda: torch.matmul(a, b))

    return {
        "size": size,
        "ms_torch":  ms_torch,
        "ms_triton": ms_triton,
        "tflops_torch":  flops / ms_torch  / 1e9,
        "tflops_triton": flops / ms_triton / 1e9,
        "speedup": ms_torch / ms_triton,   # >1 means our kernel is faster
    }

sizes = [512, 1024, 2048, 4096]
results = [benchmark(s) for s in sizes]

print(f"{'size':>6}{'torch ms':>11}{'triton ms':>11}{'torch TF/s':>12}{'triton TF/s':>13}{'speedup':>9}")
print("-" * 62)
for r in results:
    print(f"{r['size']:>6}{r['ms_torch']:>11.3f}{r['ms_triton']:>11.3f}"
          f"{r['tflops_torch']:>12.1f}{r['tflops_triton']:>13.1f}{r['speedup']:>8.2f}x")

## 8. Results - latency & throughput, our kernel vs. eager

Same math, same numbers out (section 6) - the chart is purely about speed. The left panel is
TFLOP/s (higher is better), the right is latency (lower is better). A custom tiled kernel from
scratch lands in the same range as the vendor library.

In [ ]:
import matplotlib.pyplot as plt

labels = [str(r["size"]) for r in results]
x = np.arange(len(labels))
w = 0.38

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].bar(x - w/2, [r["tflops_torch"]  for r in results], w, label="torch.matmul", color="#888")
ax[0].bar(x + w/2, [r["tflops_triton"] for r in results], w, label="triton (ours)", color="#5cb85c")
ax[0].set_title("Throughput (TFLOP/s) - higher is better")
ax[0].set_xlabel("matrix size (NxN)"); ax[0].set_xticks(x); ax[0].set_xticklabels(labels); ax[0].legend()

ax[1].bar(x - w/2, [r["ms_torch"]  for r in results], w, label="torch.matmul", color="#888")
ax[1].bar(x + w/2, [r["ms_triton"] for r in results], w, label="triton (ours)", color="#5cb85c")
ax[1].set_title("Latency (ms) - lower is better")
ax[1].set_xlabel("matrix size (NxN)"); ax[1].set_xticks(x); ax[1].set_xticklabels(labels); ax[1].legend()

plt.tight_layout(); plt.show()

avg = np.mean([r["speedup"] for r in results])
print(f">>> Our Triton kernel runs at {avg:.2f}x the speed of torch.matmul on average")
print(">>> ...computing the exact same result (verified in section 6).")

## 9. Takeaways & things to try

**What you saw:** one operation, two implementations, **identical output** - and a handful of
lines of Triton reaches the same speed range as PyTorch's cuBLAS-backed matmul. The speed came
entirely from *how* the data moves: tile the output, reuse loaded data on-chip, lay out the grid
so neighboring tiles share cache.

**Experiment (edit and re-run):**
- Change the **block sizes** `BLOCK_M / BLOCK_N / BLOCK_K` in `triton_matmul` (try `32`, `128`).
  Bigger tiles reuse more data but need more on-chip memory - find the sweet spot.
- Set **`GROUP_M=1`** to disable grouped ordering. Watch the larger sizes get slower: that is
  the L2-cache memory-access effect, isolated.
- Add a **non-square** shape (e.g. `512 x 4096 x 1024`) and confirm correctness still holds -
  the masks handle ragged edges.
- Fuse an activation: apply `tl.maximum(acc, 0)` (a ReLU) before the store. A **fused matmul +
  activation** does the same work without a second pass over memory.

**Why it matters for the edge:** kernel optimization is orthogonal to the other four ideas and
composes with them. A **quantized, fused matmul kernel** is exactly how a pruned, distilled,
int8 model actually runs fast on real hardware.